# 01 - Exploratory Data Analysis
## Credit Card Fraud Detection
**Universidad de Medellín - MLOps Final Project**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')
from src.data.preprocessing import load_data, engineer_features

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

df = load_data('../data/raw/credit_card_frauds.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Class distribution
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].mean() * 100
print(f'Fraud rate: {fraud_pct:.2f}%')
print(f'Fraud: {fraud_counts[1]:,} | Legit: {fraud_counts[0]:,}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution (count)')
axes[0].set_xticklabels(['Legitimate', 'Fraud'], rotation=0)
axes[1].pie([fraud_counts[0], fraud_counts[1]], labels=['Legitimate', 'Fraud'],
            colors=['steelblue', 'tomato'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Distribution (%)')
plt.tight_layout()
plt.savefig('../docs/eda_class_distribution.png')
plt.show()

In [ ]:
# Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[df['is_fraud']==0]['amt'].clip(upper=500).hist(bins=60, ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Transaction Amount - Legitimate')
axes[0].set_xlabel('Amount (USD)')
df[df['is_fraud']==1]['amt'].clip(upper=500).hist(bins=60, ax=axes[1], color='tomato', alpha=0.8)
axes[1].set_title('Transaction Amount - Fraud')
axes[1].set_xlabel('Amount (USD)')
plt.tight_layout()
plt.savefig('../docs/eda_amount_distribution.png')
plt.show()

print(df.groupby('is_fraud')['amt'].describe())

In [ ]:
# Feature engineering & derived features
df_eng = engineer_features(df)

# Fraud by hour
fraud_by_hour = df_eng.groupby('hour')['is_fraud'].mean() * 100
fig, ax = plt.subplots(figsize=(12, 4))
fraud_by_hour.plot(kind='bar', ax=ax, color='tomato', alpha=0.8)
ax.set_title('Fraud Rate by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('Fraud Rate (%)')
plt.tight_layout()
plt.savefig('../docs/eda_fraud_by_hour.png')
plt.show()

In [ ]:
# Fraud by category
fraud_by_cat = df.groupby('category')['is_fraud'].mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(12, 5))
fraud_by_cat.plot(kind='barh', ax=ax, color='tomato', alpha=0.8)
ax.set_title('Fraud Rate by Merchant Category')
ax.set_xlabel('Fraud Rate (%)')
plt.tight_layout()
plt.savefig('../docs/eda_fraud_by_category.png')
plt.show()

In [ ]:
# Correlation heatmap of numeric features
num_cols = ['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'is_fraud']
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../docs/eda_correlation.png')
plt.show()

## Key EDA Findings

| Finding | Detail |
|---|---|
| **Class imbalance** | Only 0.52% fraud → requires `class_weight='balanced'` or `scale_pos_weight` |
| **Amount** | Fraud transactions tend to be larger on average |
| **Hour** | Fraud peaks at late-night hours (00:00 – 03:00) |
| **Category** | `shopping_net`, `misc_net`, `grocery_net` show highest fraud rates |
| **Distance** | Fraudsters tend to be geographically far from the merchant |
| **No nulls** | Dataset is clean, no imputation needed |